# 01 · Training-only feature engineering

This notebook investigates the feature-engineering stage without turning precomputed matrices into model-evaluation evidence. It loads the exact prepared handoff from notebook 00, fits `RawFeatureScreen` and `FeatureEngineering` on **training rows only**, transforms later partitions with frozen state, and saves only the recipe that later raw-row pipelines must refit inside each CV fold.


## 1. Load the checked prepared stage


In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Locate the repository from either the project root or notebook/.
current_directory = Path.cwd().resolve()
PROJECT_ROOT = None
for directory in [current_directory, *current_directory.parents]:
    if (directory / "src" / "fraud_detection").is_dir():
        PROJECT_ROOT = directory
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the project root. Expected src/fraud_detection."
    )

source_path = str(PROJECT_ROOT / "src")
if source_path not in sys.path:
    sys.path.insert(0, source_path)

SEED = 42
SMOKE = os.environ.get("FRAUD_NOTEBOOK_SMOKE", "0") == "1"
MAX_ROWS = int(os.environ.get("FRAUD_NOTEBOOK_MAX_ROWS", "0"))
if MAX_ROWS < 0:
    raise ValueError("FRAUD_NOTEBOOK_MAX_ROWS must be nonnegative.")

stage_setting = os.environ.get(
    "FRAUD_NOTEBOOK_STAGE_DIR", "data/processed/learning_stages"
)
STAGE_ROOT = Path(stage_setting).expanduser()
if not STAGE_ROOT.is_absolute():
    STAGE_ROOT = PROJECT_ROOT / STAGE_ROOT
STAGE_ROOT = STAGE_ROOT.resolve()
STAGE_ROOT.mkdir(parents=True, exist_ok=True)

print({
    "project_root": str(PROJECT_ROOT),
    "stage_root": str(STAGE_ROOT),
    "smoke": SMOKE,
    "max_rows": MAX_ROWS,
    "seed": SEED,
})

from fraud_detection.data_preparation import load_prepared_data
from fraud_detection.data import calculate_file_sha256
from fraud_detection.utils import frame_fingerprint

pointer_path = STAGE_ROOT / "prepared_latest.json"
if not pointer_path.is_file():
    raise FileNotFoundError(
        f"Missing {pointer_path}. Run notebook 00_data_preparation.ipynb first."
    )
pointer = json.loads(pointer_path.read_text(encoding="utf-8"))

if bool(pointer.get("smoke")) != SMOKE:
    raise ValueError(
        "Notebook mode differs from notebook 00. Use the same FRAUD_NOTEBOOK_SMOKE setting."
    )
if int(pointer.get("max_rows", 0)) != MAX_ROWS:
    raise ValueError(
        "FRAUD_NOTEBOOK_MAX_ROWS differs from notebook 00. Use the same row limit."
    )

prepared_folder = Path(pointer["prepared_folder"])
prepared_data = load_prepared_data(prepared_folder)
manifest_path = prepared_folder / "prepared_manifest.json"
actual_manifest_sha256 = calculate_file_sha256(manifest_path)
if actual_manifest_sha256 != pointer["prepared_manifest_sha256"]:
    raise ValueError("Prepared manifest changed since notebook 00.")

partitions = prepared_data["partitions"]
labeled_pool = prepared_data["labeled_pool"]
kaggle_test = prepared_data["kaggle_test"]
DATA_SCOPE = prepared_data["manifest"]["data_scope"]
if DATA_SCOPE != pointer["data_scope"]:
    raise ValueError("Prepared data scope differs from notebook 00 pointer.")

current_fingerprints = {
    name: frame_fingerprint(getattr(partitions, name))
    for name in ("train", "validation", "holdout")
}
if current_fingerprints != pointer["partition_fingerprints"]:
    raise ValueError("Prepared partition fingerprints differ from notebook 00.")

SOURCE_FILE_SHA256 = dict(pointer.get("source_file_sha256", {}))
JOIN_QUALITY_REPORT = prepared_data.get("quality_report", {})

print({
    "data_scope": DATA_SCOPE,
    "prepared_folder": str(prepared_folder),
    "train_rows": len(partitions.train),
    "validation_rows": len(partitions.validation),
    "holdout_rows": len(partitions.holdout),
    "kaggle_test_rows": len(kaggle_test),
    "integrity": "checksum + shape + partition fingerprints verified",
})


## 2. Declare the exploratory engineering recipe


In [ ]:
from fraud_detection.features import FeatureEngineering, RawFeatureScreen

RAW_SCREEN_STRATEGY = os.environ.get("FRAUD_NOTEBOOK_RAW_SCREEN", "all")
TOP_V_RAW = int(os.environ.get("FRAUD_NOTEBOOK_TOP_V", "0"))
TOP_V_FEATURES = TOP_V_RAW or None
ENGINEERING_ENCODING = os.environ.get("FRAUD_NOTEBOOK_ENGINEERING_ENCODING", "legacy")
FEATURE_GROUPS = os.environ.get("FRAUD_NOTEBOOK_FEATURE_GROUPS", "all")
N_CLUSTERS = int(os.environ.get("FRAUD_NOTEBOOK_N_CLUSTERS", "0"))
MODEL_ENCODING = os.environ.get("FRAUD_NOTEBOOK_MODEL_ENCODING", "auto")

if RAW_SCREEN_STRATEGY not in RawFeatureScreen.STRATEGIES:
    raise ValueError(f"RAW screen must be one of {RawFeatureScreen.STRATEGIES}")
if ENGINEERING_ENCODING not in FeatureEngineering.ENCODINGS:
    raise ValueError(f"Engineering encoding must be one of {FeatureEngineering.ENCODINGS}")
if FEATURE_GROUPS not in FeatureEngineering.FEATURE_GROUPS:
    raise ValueError(f"Feature groups must be one of {FeatureEngineering.FEATURE_GROUPS}")
if MODEL_ENCODING not in ("auto", "legacy", "onehot"):
    raise ValueError("FRAUD_NOTEBOOK_MODEL_ENCODING must be auto, legacy or onehot")
if N_CLUSTERS < 0:
    raise ValueError("FRAUD_NOTEBOOK_N_CLUSTERS must be nonnegative")

feature_recipe = {
    "raw_screen_strategy": RAW_SCREEN_STRATEGY,
    "top_v_features": TOP_V_FEATURES,
    "feature_groups": FEATURE_GROUPS,
    "n_clusters": N_CLUSTERS,
    "encoding": MODEL_ENCODING,
}
display(pd.Series({**feature_recipe, "exploration_encoding": ENGINEERING_ENCODING}, name="engineering_recipe"))


## 3. Fit screening and engineering on training only


In [ ]:
X_train = partitions.train.drop(columns="isFraud")
y_train = partitions.train.isFraud
X_validation = partitions.validation.drop(columns="isFraud")
X_holdout = partitions.holdout.drop(columns="isFraud")

raw_screen = RawFeatureScreen(
    raw_screen_strategy=RAW_SCREEN_STRATEGY,
    top_v_features=TOP_V_FEATURES,
    seed=SEED,
).fit(X_train, y_train)

screened_train = raw_screen.transform(X_train)
screened_validation = raw_screen.transform(X_validation)
screened_holdout = raw_screen.transform(X_holdout)
screened_test = raw_screen.transform(kaggle_test)

display(pd.Series(raw_screen.screening_report_, name="raw_screen_report"))

engineer = FeatureEngineering(
    encoding=ENGINEERING_ENCODING,
    random_state=SEED,
    feature_groups=FEATURE_GROUPS,
    n_clusters=N_CLUSTERS,
).fit(screened_train, y_train)

engineered_train = engineer.transform(screened_train)
engineered_validation = engineer.transform(screened_validation)
engineered_holdout = engineer.transform(screened_holdout)
engineered_test = engineer.transform(screened_test)

shape_report = pd.DataFrame([
    {"view": "train", "rows": len(engineered_train), "features": engineered_train.shape[1]},
    {"view": "validation", "rows": len(engineered_validation), "features": engineered_validation.shape[1]},
    {"view": "holdout", "rows": len(engineered_holdout), "features": engineered_holdout.shape[1]},
    {"view": "kaggle_test", "rows": len(engineered_test), "features": engineered_test.shape[1]},
])
display(shape_report)


## 4. Inspect fitted training state and generated schema


In [ ]:
display(pd.Series(engineer.amount_stats_, name="training_amount_statistics"))
print("resolved feature_groups:", engineer.feature_groups_)
print("resolved encoding:", engineer.encoding_)
print("engineered feature count:", len(engineer.output_features_))
display(pd.DataFrame({"feature": engineer.output_features_}).head(40))

# The transformed schema must be frozen across later data.
assert list(engineered_train.columns) == list(engineered_validation.columns)
assert list(engineered_train.columns) == list(engineered_holdout.columns)
assert list(engineered_train.columns) == list(engineered_test.columns)
assert np.isfinite(engineered_train.select_dtypes(include=[np.number]).replace([np.inf, -np.inf], np.nan).notna().sum()).all()


## 5. Verify row-wise inference equivalence


In [ ]:
# A fitted transformer must give the same result for a row whether transformed
# alone or inside a batch. This catches accidental transform-time learning.
row = screened_validation.iloc[[0]].copy()
row_alone = engineer.transform(row)
row_in_batch = engineer.transform(screened_validation.iloc[:5]).iloc[[0]]
pd.testing.assert_frame_equal(
    row_alone.reset_index(drop=True),
    row_in_batch.reset_index(drop=True),
    check_dtype=True,
)
print("single-row vs batch transformation: passed")


## 6. Save the feature recipe for notebook 02


In [ ]:
feature_recipe_record = {
    "schema_version": 1,
    "prepared_manifest_sha256": pointer["prepared_manifest_sha256"],
    "partition_fingerprints": pointer["partition_fingerprints"],
    "data_scope": DATA_SCOPE,
    "seed": SEED,
    **feature_recipe,
    "exploration_encoding": ENGINEERING_ENCODING,
    "raw_screen_report": raw_screen.screening_report_,
    "engineered_feature_count": int(engineered_train.shape[1]),
    "engineered_features": list(engineered_train.columns),
}
recipe_path = STAGE_ROOT / "feature_recipe.json"
recipe_path.write_text(json.dumps(feature_recipe_record, indent=2, default=str), encoding="utf-8")
print("Saved:", recipe_path)
print("Next: run 02_feature_selection.ipynb")


## Important interpretation

The fitted `engineer` object above is for learning and diagnostics only. Notebook 03 does **not** train models from these precomputed matrices. Every model/CV fold starts again from raw rows and refits screening, feature engineering, feature selection and preprocessing inside the complete pipeline.
